# Goal

**Daegu Traffic Accident Damage Prediction AI Contest**

이동수단의 발달에 따라 다양한 유형의 교통사고들이 계속 발생하고 있습니다.

한국자동차연구원과 대구디지털혁신진흥원에서는 해당 사고의 원인을 규명하고 사고율을 낮추기 위해,

시공간 정보로부터 사고위험도(ECLO)를 예측하는 AI 알고리즘 발굴을 목표로 본 대회를 개최합니다.



※ ECLO(Equivalent Casualty Loss Only) : 인명피해 심각도

**ECLO = 사망자수 * 10 + 중상자수 * 5 + 경상자수 * 3 + 부상자수 * 1**

본 대회에서는 사고의 위험도를 인명피해 심각도로 측정


# Data Description

Dataset Info.

train.csv [파일]
- ID : 대구에서 발생한 교통사고의 고유 ID
-2019년부터 2021년까지의 교통사고 데이터로 구성
- 해당 사고가 발생한 당시의 시공간 정보와 사고 관련 정보 포함
- ECLO : 인명피해 심각도


test.csv [파일]
- ID : 대구에서 발생한 교통사고의 고유 ID
- 2022년도의 교통사고 데이터로 구성
- 추론 시점에서 획득할 수 있는 정보로 구성


sample_submission.csv [파일] - 제출 양식
- ID : 추론 샘플의 고유 ID
- ECLO : 예측한 인명피해 심각도


대구 빅데이터 마트 데이터 [폴더]
- 대구 빅데이터활용센터에서 구축한 빅데이터 마트 데이터 중 제공 가능한 일부 데이터셋
- 상세한 명세는 폴더 내부의 빅데이터 마트 데이터 설명서.hwp 참고
- 전체 빅데이터 마트 데이터셋을 활용하기 위해서는 대구 빅데이터활용센터 방문 필요


countrywide_accident.csv [파일]
- 대구를 제외한 전국에서 발생한 교통사고 데이터
- 2019년부터 2021년까지의 교통사고 데이터로 구성
- train.csv와 양식 동일


대구 보안등 정보.csv [파일]
- 대구에 존재하는 보안등 관련 정보


대구 어린이 보호 구역 정보.csv [파일]
- 대구에 존재하는 어린이 보호 구역 관련 정보


대구 주차장 정보.csv [파일]
- 대구에 존재하는 주차장 관련 정보


대구 CCTV 정보.csv [파일]
- 대구에 존재하는 CCTV 관련 정보

# Library

In [ ]:
#Regression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib import rc
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

from sklearn.model_selection import KFold, cross_val_score, train_test_split, TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor, StackingRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder,OrdinalEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.inspection import PartialDependenceDisplay
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna

#automl
from pycaret.regression import *

ModuleNotFoundError: No module named 'catboost'

# Datset

In [ ]:
path = '/Users/baegnamjin/Desktop/Dataset/Daegu_Accident'

train = pd.read_csv(path + '/train.csv')
test = pd.read_csv(path + '/test.csv')
ss = pd.read_csv(path + '/sample_submission.csv')

train.drop(['ID'],axis=1, inplace=True)
test.drop(['ID'],axis=1, inplace=True)
#external

# Preprocessing

In [ ]:
y = train['ECLO']
train = train[test.columns]
train['ECLO'] = y
train.head()

,사고일시,요일,기상상태,시군구,도로형태,노면상태,사고유형,ECLO
0,2019-01-01 00,화요일,맑음,대구광역시 중구 대신동,단일로 - 기타,건조,차대사람,5
1,2019-01-01 00,화요일,흐림,대구광역시 달서구 감삼동,단일로 - 기타,건조,차대사람,3
2,2019-01-01 01,화요일,맑음,대구광역시 수성구 두산동,단일로 - 기타,건조,차대사람,3
3,2019-01-01 02,화요일,맑음,대구광역시 북구 복현동,단일로 - 기타,건조,차대차,5
4,2019-01-01 04,화요일,맑음,대구광역시 동구 신암동,단일로 - 기타,건조,차대차,3


In [ ]:
def preprocessing(df):
    df['구'] = df['시군구'].str.split(" ",expand=True)[1]
    df['동'] = df['시군구'].str.split(" ",expand=True)[2]

    df['년도'] = pd.to_datetime(df['사고일시']).dt.year
    df['월'] = pd.to_datetime(df['사고일시']).dt.month
    df['일'] = pd.to_datetime(df['사고일시']).dt.day
    df['시간'] = pd.to_datetime(df['사고일시']).dt.hour

    return df

train = preprocessing(train)
test = preprocessing(test)

# Modeling

In [ ]:
train.drop(['사고일시','시군구'],axis=1,inplace=True)
test.drop(['사고일시','시군구'],axis=1,inplace=True)

In [ ]:
num_cols = train.select_dtypes(exclude='object').columns.tolist()
cat_cols = train.select_dtypes(include='object').columns.tolist()

In [ ]:
# #1. Label Encoding

# encoder = LabelEncoder()
# for col in cat_cols:
#     train[col] = encoder.fit_transform(train[col])
#     test[col] = encoder.transform(test[col])

# Feature Engineering

In [ ]:
X = train.drop(['ECLO'], axis=1)
y = train['ECLO']

from openfe import OpenFE, transform
ofe = OpenFE()
features = ofe.fit(data=X, label=y)

The number of candidate features is 539
Start stage I selection.


100%|█████████████████████████████████████████████████████████████████| 4/4 [00:12<00:00,  3.04s/it]


219 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|█████████████████████████████████████████████████████████████████| 4/4 [00:43<00:00, 10.78s/it]


The number of remaining candidate features is 282
Start stage II selection.


100%|█████████████████████████████████████████████████████████████████| 4/4 [00:32<00:00,  8.02s/it]


Finish data processing.


In [ ]:
X, test = transform(X, test, features, n_jobs=4)
X.shape,test.shape

((39609, 293), (10963, 293))

In [ ]:
X['ECLO'] = y
train = X

# Split

In [ ]:
train_df, valid_df = train_test_split(train,test_size=.3, random_state=42)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_log_error, mean_squared_error

def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))


In [ ]:
from autogluon.core.metrics import make_scorer
score = make_scorer(name='RMSLE',
                                 score_func=rmsle,
                                 optimum=1,
                                 greater_is_better=False)

In [ ]:
# which models are good?

from autogluon.tabular import TabularDataset, TabularPredictor

data = TabularDataset(train_df)
model = TabularPredictor(label='ECLO', problem_type='regression', eval_metric=score)

No path specified. Models will be saved in: "AutogluonModels/ag-20231122_103801/"


In [ ]:
# hyperparameters={
#         'GBM': {},
#         'CAT': {},
#         'XGB' : {},
#         "NN_TORCH" : {}
#     }
model.fit(train_df, hyperparameters={
        'GBM': {},
        'CAT': {},
        'XGB' : {},
        "NN_TORCH" : {}
    })

	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "AutogluonModels/ag-20231122_103801/"
AutoGluon Version:  0.8.2
Python Version:     3.10.0
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 23.1.0: Mon Oct  9 21:28:12 PDT 2023; root:xnu-10002.41.9~6/RELEASE_ARM64_T8103
Disk Space Avail:   172.10 GB / 245.11 GB (70.2%)
Train Data Rows:    27726
Train Data Columns: 293
Label Column: ECLO
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    2510.44 MB
	Train Data (Original)  Memory Usage: 73.38 MB (2.9% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify s

In [ ]:
#no FE

model.leaderboard(valid_df)

                  model  score_test  score_val  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   -0.447542  -0.436070        0.060951       0.018086   9.191725                 0.001768                0.000233           0.167187            2       True         12
1               XGBoost   -0.448795  -0.437153        0.021199       0.006322   0.616936                 0.021199                0.006322           0.616936            1       True          9
2        NeuralNetTorch   -0.453429  -0.442089        0.027334       0.007359   7.697644                 0.027334                0.007359           7.697644            1       True         10
3              CatBoost   -0.463325  -0.453328        0.031692       0.004078   2.718370                 0.031692                0.004078           2.718370            1       True          6
4       NeuralNetFastAI   -0.463555  -0.

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.447542,-0.436070,0.060951,0.018086,9.191725,0.001768,0.000233,0.167187,2,True,12
1,XGBoost,-0.448795,-0.437153,0.021199,0.006322,0.616936,0.021199,0.006322,0.616936,1,True,9
2,NeuralNetTorch,-0.453429,-0.442089,0.027334,0.007359,7.697644,0.027334,0.007359,7.697644,1,True,10
3,CatBoost,-0.463325,-0.453328,0.031692,0.004078,2.718370,0.031692,0.004078,2.718370,1,True,6
4,NeuralNetFastAI,-0.463555,-0.454715,0.087007,0.016244,11.946449,0.087007,0.016244,11.946449,1,True,8
5,LightGBMXT,-0.463658,-0.454157,0.005168,0.003197,0.821281,0.005168,0.003197,0.821281,1,True,3
6,LightGBM,-0.464338,-0.455183,0.010650,0.004172,0.709958,0.010650,0.004172,0.709958,1,True,4
7,LightGBMLarge,-0.464582,-0.456491,0.018499,0.004010,0.887120,0.018499,0.004010,0.887120,1,True,11
8,ExtraTreesMSE,-0.488133,-0.480402,0.280865,0.045359,1.278898,0.280865,0.045359,1.278898,1,True,7
9,RandomForestMSE,-0.490688,-0.485161,0.309131,0.055213,3.197742,0.309131,0.055213,3.197742,1,True,5


In [ ]:
#4개 모델만
model.leaderboard(valid_df)

                 model  score_test  score_val  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   -0.448077  -0.436554        0.437096       0.126513  30.444714                 0.001418                0.000226           0.064315            2       True          5
1              XGBoost   -0.449196  -0.437074        0.155651       0.045130   6.450487                 0.155651                0.045130           6.450487            1       True          3
2       NeuralNetTorch   -0.455234  -0.446028        0.252346       0.068162  21.649510                 0.252346                0.068162          21.649510            1       True          4
3             CatBoost   -0.463518  -0.453277        0.070457       0.027702  18.713072                 0.070457                0.027702          18.713072            1       True          2
4             LightGBM   -0.465397  -0.455957

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.448077,-0.436554,0.437096,0.126513,30.444714,0.001418,0.000226,0.064315,2,True,5
1,XGBoost,-0.449196,-0.437074,0.155651,0.045130,6.450487,0.155651,0.045130,6.450487,1,True,3
2,NeuralNetTorch,-0.455234,-0.446028,0.252346,0.068162,21.649510,0.252346,0.068162,21.649510,1,True,4
3,CatBoost,-0.463518,-0.453277,0.070457,0.027702,18.713072,0.070457,0.027702,18.713072,1,True,2
4,LightGBM,-0.465397,-0.455957,0.027681,0.012995,2.280402,0.027681,0.012995,2.280402,1,True,1


In [ ]:
pred = model.predict(test)

In [ ]:
ss['ECLO'] = pred
path = "./daegu_autogluon_drop_rigion_openfe.csv"
ss.to_csv(path,index=False)